In [2]:
%pip install pixeltable-pgserver
%pip install psycopg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 41.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pixeltable-pgserver]


In [3]:
import os
from urllib.request import urlretrieve
import pandas as pd
import pixeltable_pgserver as pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw7_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

/usr/local/lib/python3.13/dist-packages/platformdirs/api.py:459: RuntimeDirWarning: XDG_RUNTIME_DIR is not set, falling back to /tmp/runtime-0
  return Path(self.user_runtime_dir)


PostgreSQL 16.14 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 14.2.1 20250110 (Red Hat 14.2.1-11), 64-bit


In [4]:
%load_ext sql
%sql postgresql://postgres:@/postgres?host=/tmp/hw7_pg
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

# Завдання 1. Setup і завантаження даних NYC TLC Yellow Taxi Trip Records

In [5]:
URL = (
    'https://d37ci6vzurychx.cloudfront.net/trip-data/'
    'yellow_tripdata_2024-03.parquet'
)

parquet_path = '/content/yellow_tripdata_2024-03.parquet'
urlretrieve(URL, parquet_path)

columns = [
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'trip_distance',
    'fare_amount',
    'passenger_count',
]

df = pd.read_parquet(parquet_path, columns=columns).head(500_000)

# Basic cleaning before loading to PostgreSQL
df = df[(df['fare_amount'] > 0) & (df['fare_amount'] < 500)]
df = df[(df['trip_distance'] > 0) & (df['trip_distance'] < 100)]
df = df[df['PULocationID'].notna()]

df.to_sql(
    'nyc_taxi_raw',
    engine,
    if_exists='replace',
    index=False,
    chunksize=10_000,
)

pd.read_sql('SELECT COUNT(*) AS n_rows FROM nyc_taxi_raw', engine)


,n_rows
0,486964


Робимо daily aggregation t7_hw_taxi_daily:

In [6]:
%%sql
DROP TABLE IF EXISTS t7_hw_taxi_daily CASCADE;

CREATE TABLE t7_hw_taxi_daily AS
SELECT
    "PULocationID"::INTEGER AS pickup_zone_id,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE AS ts_day,
    COUNT(*)::INTEGER AS trip_count,
    AVG(fare_amount)::NUMERIC(10, 2) AS avg_fare,
    SUM(trip_distance)::NUMERIC(12, 2) AS total_distance
FROM nyc_taxi_raw
WHERE fare_amount > 0
  AND fare_amount < 500
  AND trip_distance > 0
  AND trip_distance < 100
  AND "PULocationID" IS NOT NULL
GROUP BY
    "PULocationID"::INTEGER,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE;

ALTER TABLE t7_hw_taxi_daily
    ADD PRIMARY KEY (pickup_zone_id, ts_day);

SELECT pickup_zone_id, COUNT(*) AS n_days
FROM t7_hw_taxi_daily
GROUP BY pickup_zone_id
ORDER BY pickup_zone_id
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
Done.
1217 rows affected.
Done.
10 rows affected.


pickup_zone_id,n_days
1,5
3,6
4,6
6,4
7,6
8,1
9,5
10,6
11,4
12,5


Демонстрація production-style з моментом часу, в окремій typed-таблиці з TIMESTAMPTZ:

In [7]:
%%sql
DROP TABLE IF EXISTS t7_hw_taxi_trips_typed;

CREATE TABLE t7_hw_taxi_trips_typed AS
SELECT
    "PULocationID"::INTEGER AS pickup_zone_id,
    "DOLocationID"::INTEGER AS dropoff_zone_id,
    (tpep_pickup_datetime::TIMESTAMP AT TIME ZONE 'America/New_York') AS pickup_at,
    (tpep_dropoff_datetime::TIMESTAMP AT TIME ZONE 'America/New_York') AS dropoff_at,
    trip_distance::NUMERIC(10, 2) AS trip_distance,
    fare_amount::NUMERIC(10, 2) AS fare_amount,
    passenger_count::INTEGER AS passenger_count
FROM nyc_taxi_raw
WHERE fare_amount > 0
  AND fare_amount < 500
  AND trip_distance > 0
  AND trip_distance < 100;

SELECT pickup_at, pg_typeof(pickup_at) AS pickup_type
FROM t7_hw_taxi_trips_typed
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
Done.
486964 rows affected.
10 rows affected.


pickup_at,pickup_type
2024-03-01 05:18:51+00:00,timestamp with time zone
2024-03-01 05:26:00+00:00,timestamp with time zone
2024-03-01 05:09:22+00:00,timestamp with time zone
2024-03-01 05:33:45+00:00,timestamp with time zone
2024-03-01 05:05:43+00:00,timestamp with time zone
2024-03-01 05:50:42+00:00,timestamp with time zone
2024-03-01 05:08:23+00:00,timestamp with time zone
2024-03-01 05:24:58+00:00,timestamp with time zone
2024-03-01 05:49:40+00:00,timestamp with time zone
2024-03-01 05:21:43+00:00,timestamp with time zone


# Завдання 2. Time-series ML features

Feature 1. Lag features: lag-1d, lag-7d, lag-30d

In [8]:
%%sql
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    LAG(trip_count, 1)  OVER w AS trips_lag_1d,
    LAG(trip_count, 7)  OVER w AS trips_lag_7d,
    LAG(trip_count, 30) OVER w AS trips_lag_30d
FROM t7_hw_taxi_daily
WINDOW w AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;


 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
20 rows affected.


pickup_zone_id,ts_day,trip_count,trips_lag_1d,trips_lag_7d,trips_lag_30d
1,2024-03-01,1,None,None,None
1,2024-03-02,2,1,None,None
1,2024-03-03,2,2,None,None
1,2024-03-04,1,2,None,None
1,2024-03-05,1,1,None,None
3,2024-03-01,3,None,None,None
3,2024-03-02,1,3,None,None
3,2024-03-03,4,1,None,None
3,2024-03-04,2,4,None,None
3,2024-03-05,1,2,None,None


Feature 2. Rolling 7-day average fare

In [9]:
%%sql
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
    COUNT(*)      OVER w7 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w7 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;


 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
20 rows affected.


pickup_zone_id,ts_day,avg_fare,rolling_avg_fare_7d,days_in_window
1,2024-03-01,105.00,105.0000000000000000,1
1,2024-03-02,95.00,100.0000000000000000,2
1,2024-03-03,51.00,83.6666666666666667,3
1,2024-03-04,40.00,72.7500000000000000,4
1,2024-03-05,120.00,82.2000000000000000,5
3,2024-03-01,39.17,39.1700000000000000,1
3,2024-03-02,21.50,30.3350000000000000,2
3,2024-03-03,25.03,28.5666666666666667,3
3,2024-03-04,39.15,31.2125000000000000,4
3,2024-03-05,45.50,34.0700000000000000,5


Feature 3. Rolling 30-day fare volatility

In [10]:
%%sql
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
    COUNT(*)              OVER w30 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w30 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;


 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
20 rows affected.


pickup_zone_id,ts_day,avg_fare,rolling_std_fare_30d,days_in_window
1,2024-03-01,105.00,None,1
1,2024-03-02,95.00,7.0710678118654752,2
1,2024-03-03,51.00,28.7286152352203244,3
1,2024-03-04,40.00,32.0455405113826511,4
1,2024-03-05,120.00,34.8812270426371325,5
3,2024-03-01,39.17,None,1
3,2024-03-02,21.50,12.4945768235662948,2
3,2024-03-03,25.03,9.3508413168726873,3
3,2024-03-04,39.15,9.2894505578460703,4
3,2024-03-05,45.50,10.2736045281098883,5


Feature 4. Expanding cumulative trips

In [11]:
%%sql
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    SUM(trip_count) OVER we AS cumulative_trips,
    AVG(trip_count) OVER we AS expanding_avg_trips
FROM t7_hw_taxi_daily
WINDOW we AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;


 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
20 rows affected.


pickup_zone_id,ts_day,trip_count,cumulative_trips,expanding_avg_trips
1,2024-03-01,1,1,1.00000000000000000000
1,2024-03-02,2,3,1.5000000000000000
1,2024-03-03,2,5,1.6666666666666667
1,2024-03-04,1,6,1.5000000000000000
1,2024-03-05,1,7,1.4000000000000000
3,2024-03-01,3,3,3.0000000000000000
3,2024-03-02,1,4,2.0000000000000000
3,2024-03-03,4,8,2.6666666666666667
3,2024-03-04,2,10,2.5000000000000000
3,2024-03-05,1,11,2.2000000000000000


Combined feature view:

In [12]:
%%sql
CREATE OR REPLACE VIEW v_t7_hw_taxi_features AS
WITH features_basic AS (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        avg_fare,
        total_distance,

        LAG(trip_count, 1)  OVER w_pid AS trips_lag_1d,
        LAG(trip_count, 7)  OVER w_pid AS trips_lag_7d,
        LAG(trip_count, 30) OVER w_pid AS trips_lag_30d,

        AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
        STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,

        SUM(trip_count) OVER we AS cumulative_trips,
        AVG(trip_count) OVER we AS expanding_avg_trips
    FROM t7_hw_taxi_daily
    WINDOW
        w_pid AS (
            -- PARTITION BY pickup_zone_id
            ORDER BY ts_day
        ),
        w7 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
        ),
        w30 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
        ),
        we AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
)
SELECT *
FROM features_basic
WHERE trips_lag_30d IS NOT NULL;

SELECT *
FROM v_t7_hw_taxi_features
ORDER BY pickup_zone_id, ts_day
LIMIT 20;


 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
Done.
20 rows affected.


pickup_zone_id,ts_day,trip_count,avg_fare,total_distance,trips_lag_1d,trips_lag_7d,trips_lag_30d,rolling_avg_fare_7d,rolling_std_fare_30d,cumulative_trips,expanding_avg_trips
1,2024-03-01,1,105.00,0.04,2682,1,3463,105.0000000000000000,None,1,1.00000000000000000000
1,2024-03-02,2,95.00,18.43,10,2,18,100.0000000000000000,7.0710678118654752,3,1.5000000000000000
1,2024-03-03,2,51.00,0.09,308,13,3107,83.6666666666666667,28.7286152352203244,5,1.6666666666666667
1,2024-03-04,1,40.00,0.01,31,4,96,72.7500000000000000,32.0455405113826511,6,1.5000000000000000
1,2024-03-05,1,120.00,0.03,1449,4,1091,82.2000000000000000,34.8812270426371325,7,1.4000000000000000
3,2024-03-01,3,39.17,34.00,5,5306,26,39.1700000000000000,None,3,3.0000000000000000
3,2024-03-02,1,21.50,1.80,1,1,8,30.3350000000000000,12.4945768235662948,4,2.0000000000000000
3,2024-03-03,4,25.03,24.13,2,14,1,28.5666666666666667,9.3508413168726873,8,2.6666666666666667
3,2024-03-04,2,39.15,16.86,483,26,12,31.2125000000000000,9.2894505578460703,10,2.5000000000000000
3,2024-03-05,1,45.50,13.90,8,3,499,34.0700000000000000,10.2736045281098883,11,2.2000000000000000


# Завдання 3. JSONB-таблиця для подій

Створюємо синтетичну таблицю t7_hw_events для ML event-log:

In [13]:
%%sql
DROP TABLE IF EXISTS t7_hw_events;

CREATE TABLE t7_hw_events (
    event_id    INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    user_id     INTEGER NOT NULL,
    event_ts    TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    event_type  TEXT NOT NULL,
    payload     JSONB NOT NULL
);

INSERT INTO t7_hw_events (user_id, event_ts, event_type, payload)
SELECT
    (n % 1000) + 1 AS user_id,
    TIMESTAMPTZ '2024-04-01 00:00:00+00'
        - ((n % 60) || ' days')::INTERVAL AS event_ts,
    CASE (n % 5)
        WHEN 0 THEN 'page_view'
        WHEN 1 THEN 'click'
        WHEN 2 THEN 'add_to_cart'
        WHEN 3 THEN 'checkout'
        ELSE        'recommendation_shown'
    END AS event_type,
    CASE (n % 5)
        WHEN 0 THEN jsonb_build_object(
            'url', '/product/' || ((n % 200) + 1),
            'referrer', (ARRAY['google','direct','email','fb'])[(n % 4) + 1],
            'device', (ARRAY['mobile','desktop','tablet'])[(n % 3) + 1]
        )
        WHEN 1 THEN jsonb_build_object(
            'element_id', 'btn_' || ((n % 50) + 1),
            'page_url', '/category/' || ((n % 20) + 1),
            'device', (ARRAY['mobile','desktop'])[(n % 2) + 1]
        )
        WHEN 2 THEN jsonb_build_object(
            'product_id', (n % 200) + 1,
            'price', ((n % 100) + 10)::NUMERIC(10, 2),
            'quantity', (n % 5) + 1,
            'tags', jsonb_build_array(
                (ARRAY['promo','new','clearance'])[(n % 3) + 1]
            )
        )
        WHEN 3 THEN jsonb_build_object(
            'order_id', 'o_' || (n + 100000),
            'total', ((n % 500) + 50)::NUMERIC(10, 2),
            'items_count', (n % 10) + 1,
            'payment_method', (ARRAY['card','paypal','crypto'])[(n % 3) + 1]
        )
        ELSE jsonb_build_object(
            'model_name', 'rec_v' || ((n % 5) + 1),
            'model_version', '1.' || (n % 10),
            'recommendations', jsonb_build_array(
                (n % 200) + 1,
                ((n + 1) % 200) + 1,
                ((n + 2) % 200) + 1
            ),
            'experiment', 'exp_' || ((n % 3) + 1)
        )
    END AS payload
FROM generate_series(1, 50000) AS n;

ANALYZE t7_hw_events;

SELECT event_type, COUNT(*) AS n_events
FROM t7_hw_events
GROUP BY event_type
ORDER BY event_type;


 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
Done.
Done.
50000 rows affected.
Done.
5 rows affected.


event_type,n_events
add_to_cart,10000
checkout,10000
click,10000
page_view,10000
recommendation_shown,10000


JSONB Query 1. Containment @>

In [14]:
%%sql
SELECT event_id, user_id, event_ts, payload
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND payload @> jsonb_build_object('payment_method', 'card')
LIMIT 10;


 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
10 rows affected.


event_id,user_id,event_ts,payload
3,4,2024-03-29 00:00:00+00:00,"{'total': 53.0, 'order_id': 'o_100003', 'items_count': 4, 'payment_method': 'card'}"
18,19,2024-03-14 00:00:00+00:00,"{'total': 68.0, 'order_id': 'o_100018', 'items_count': 9, 'payment_method': 'card'}"
33,34,2024-02-28 00:00:00+00:00,"{'total': 83.0, 'order_id': 'o_100033', 'items_count': 4, 'payment_method': 'card'}"
48,49,2024-02-13 00:00:00+00:00,"{'total': 98.0, 'order_id': 'o_100048', 'items_count': 9, 'payment_method': 'card'}"
63,64,2024-03-29 00:00:00+00:00,"{'total': 113.0, 'order_id': 'o_100063', 'items_count': 4, 'payment_method': 'card'}"
78,79,2024-03-14 00:00:00+00:00,"{'total': 128.0, 'order_id': 'o_100078', 'items_count': 9, 'payment_method': 'card'}"
93,94,2024-02-28 00:00:00+00:00,"{'total': 143.0, 'order_id': 'o_100093', 'items_count': 4, 'payment_method': 'card'}"
108,109,2024-02-13 00:00:00+00:00,"{'total': 158.0, 'order_id': 'o_100108', 'items_count': 9, 'payment_method': 'card'}"
123,124,2024-03-29 00:00:00+00:00,"{'total': 173.0, 'order_id': 'o_100123', 'items_count': 4, 'payment_method': 'card'}"
138,139,2024-03-14 00:00:00+00:00,"{'total': 188.0, 'order_id': 'o_100138', 'items_count': 9, 'payment_method': 'card'}"


JSONB Query 2. Key existence ? та ?|

In [15]:
%%sql
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ? 'tags'
LIMIT 10;

SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ?| ARRAY['experiment', 'model_version']
LIMIT 10;


 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
10 rows affected.
10 rows affected.


event_id,event_type,payload
4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [5, 6, 7]}"
9,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [10, 11, 12]}"
14,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [15, 16, 17]}"
19,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [20, 21, 22]}"
24,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [25, 26, 27]}"
29,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [30, 31, 32]}"
34,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [35, 36, 37]}"
39,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [40, 41, 42]}"
44,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [45, 46, 47]}"
49,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [50, 51, 52]}"


JSONB Query 3. JSONPath

In [16]:
%%sql
SELECT
    event_id,
    jsonb_path_query(payload, '$.recommendations[*]') AS recommended_pid
FROM t7_hw_events
WHERE event_type = 'recommendation_shown'
LIMIT 15;

SELECT
    event_id,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND jsonb_path_exists(payload, '$.total ? (@ > 300)')
LIMIT 10;


 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
15 rows affected.
10 rows affected.


event_id,order_id,total
253,o_100253,303.00
258,o_100258,308.00
263,o_100263,313.00
268,o_100268,318.00
273,o_100273,323.00
278,o_100278,328.00
283,o_100283,333.00
288,o_100288,338.00
293,o_100293,343.00
298,o_100298,348.00


# Завдання 4. GIN-index demo + EXPLAIN ANALYZE

Step 1. Containment query без GIN-індексу

In [17]:
%%sql
DROP INDEX IF EXISTS idx_t7_events_payload_gin_pathops;
DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops;
DROP INDEX IF EXISTS idx_t7_events_device_btree;

ANALYZE t7_hw_events;

EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card');


 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
Done.
Done.
Done.
Done.
10 rows affected.


QUERY PLAN
Aggregate (cost=1758.21..1758.22 rows=1 width=8) (actual time=47.796..47.798 rows=1 loops=1)
Buffers: shared hit=1000
-> Seq Scan on t7_hw_events (cost=0.00..1750.00 rows=3286 width=0) (actual time=0.012..47.561 rows=3334 loops=1)
"Filter: (payload @> jsonb_build_object('payment_method', 'card'))"
Rows Removed by Filter: 46666
Buffers: shared hit=1000
Planning:
Buffers: shared hit=5
Planning Time: 0.155 ms
Execution Time: 47.818 ms


Step 2. GIN-індекс для containment / JSONPath workload

In [18]:
%%sql
CREATE INDEX idx_t7_events_payload_gin_pathops
    ON t7_hw_events USING GIN (payload jsonb_path_ops);

ANALYZE t7_hw_events;

SELECT pg_size_pretty(pg_relation_size('idx_t7_events_payload_gin_pathops'))
       AS gin_index_size;


 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
Done.
Done.
1 rows affected.


gin_index_size
1144 kB


Step 3. Той самий containment query після GIN

In [19]:
%%sql
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card');

EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('referrer', 'google');


 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
13 rows affected.
13 rows affected.


QUERY PLAN
Aggregate (cost=1063.65..1063.66 rows=1 width=8) (actual time=7.169..7.170 rows=1 loops=1)
Buffers: shared hit=1004
-> Bitmap Heap Scan on t7_hw_events (cost=24.56..1058.07 rows=2234 width=0) (actual time=0.660..6.893 rows=2500 loops=1)
"Recheck Cond: (payload @> jsonb_build_object('referrer', 'google'))"
Heap Blocks: exact=1000
Buffers: shared hit=1004
-> Bitmap Index Scan on idx_t7_events_payload_gin_pathops (cost=0.00..24.00 rows=2234 width=0) (actual time=0.465..0.466 rows=2500 loops=1)
"Index Cond: (payload @> jsonb_build_object('referrer', 'google'))"
Buffers: shared hit=4
Planning:


Step 4. Коли GIN не допомагає: path-specific text equality

In [20]:
%%sql
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile';


 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
8 rows affected.


QUERY PLAN
Aggregate (cost=1750.62..1750.63 rows=1 width=8) (actual time=15.102..15.103 rows=1 loops=1)
Buffers: shared hit=1000
-> Seq Scan on t7_hw_events (cost=0.00..1750.00 rows=250 width=0) (actual time=0.021..14.434 rows=8333 loops=1)
Filter: ((payload ->> 'device'::text) = 'mobile'::text)
Rows Removed by Filter: 41667
Buffers: shared hit=1000
Planning Time: 0.090 ms
Execution Time: 15.132 ms


Тепер створюємо expression index і повторюємо запит:

In [87]:
%%sql
CREATE INDEX idx_t7_events_device_btree
    ON t7_hw_events ((payload ->> 'device'));

ANALYZE t7_hw_events;

EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile';


 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
Done.
Done.
13 rows affected.


QUERY PLAN
Aggregate (cost=1243.38..1243.39 rows=1 width=8) (actual time=5.815..5.817 rows=1 loops=1)
Buffers: shared hit=1000 read=9
-> Bitmap Heap Scan on t7_hw_events (cost=97.08..1222.48 rows=8360 width=0) (actual time=0.959..4.789 rows=8333 loops=1)
Recheck Cond: ((payload ->> 'device'::text) = 'mobile'::text)
Heap Blocks: exact=1000
Buffers: shared hit=1000 read=9
-> Bitmap Index Scan on idx_t7_events_device_btree (cost=0.00..94.99 rows=8360 width=0) (actual time=0.734..0.735 rows=8333 loops=1)
Index Cond: ((payload ->> 'device'::text) = 'mobile'::text)
Buffers: shared read=9
Planning:


Порівняльна характеристика чотирьох EXPLAIN-output:

\begin{array}{|c|c|c|c|}
\hline
\text
Метрика\ / Параметр & Containment\ до\ GIN & Containment\ після\ GIN & Path-equality\ без\ Expr Index & Path-equality\ з\ Expr Index \\
\hline
Команда & @> & @> & ->> та = & ->> та = \\
Метод\ зчитування & Seq\ Scan & Bitmap\ Index\ Scan & Seq\ Scan &
Bitmap\ Index\ Scan  \\
Оцінка\ строк\ (rows) & 3286 (план) / 3 334 (факт) & 2234 (план) / 2500 (факт) & 250 (план) / 8 333 (факт!) & 8 360 (план) / 8 333 (факт)\\
Buffers\ shared & hit=1000 & 	hit=1004\ (+4\ індекси) & hit=1000 & hit=1000 + read=9\ (диск) \\
Execution\ Time & 47.818\ ms & 	7.196\ ms & 15.132\ ms & 5.849\ ms \\
Total\ Cost & 1758.22 & 	1063.66 & 1750.63 & 1243.39 \\
\hline
\end{array}

# Завдання 5. JSONB vs normalized

Step 1. Нормалізована таблиця для checkout-подій

In [21]:
%%sql
DROP TABLE IF EXISTS t7_hw_checkouts_norm;

CREATE TABLE t7_hw_checkouts_norm (
    event_id        INTEGER PRIMARY KEY,
    user_id         INTEGER NOT NULL,
    event_ts        TIMESTAMPTZ NOT NULL,
    order_id        TEXT NOT NULL,
    total           NUMERIC(10, 2) NOT NULL,
    items_count     INTEGER NOT NULL CHECK (items_count > 0),
    payment_method  TEXT NOT NULL CHECK (payment_method IN ('card', 'paypal', 'crypto'))
);

INSERT INTO t7_hw_checkouts_norm (
    event_id,
    user_id,
    event_ts,
    order_id,
    total,
    items_count,
    payment_method
)
SELECT
    event_id,
    user_id,
    event_ts,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total,
    (payload ->> 'items_count')::INTEGER AS items_count,
    payload ->> 'payment_method' AS payment_method
FROM t7_hw_events
WHERE event_type = 'checkout';

CREATE INDEX idx_t7_checkouts_norm_pm
    ON t7_hw_checkouts_norm (payment_method);

ANALYZE t7_hw_checkouts_norm;

SELECT COUNT(*) AS n_checkouts
FROM t7_hw_checkouts_norm;


 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
Done.
Done.
10000 rows affected.
Done.
Done.
1 rows affected.


n_checkouts
10000


Step 2. Один запит двома способами.   
**JSONB-версія:**

In [22]:
%%sql
EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payload ->> 'payment_method' AS payment_method,
    AVG((payload ->> 'total')::NUMERIC) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_events
WHERE event_type = 'checkout'
GROUP BY payload ->> 'payment_method'
ORDER BY avg_total DESC
LIMIT 3;


 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
18 rows affected.


QUERY PLAN
Limit (cost=1955.93..1955.93 rows=3 width=72) (actual time=14.354..14.356 rows=3 loops=1)
Buffers: shared hit=1000
-> Sort (cost=1955.93..1969.84 rows=5565 width=72) (actual time=14.352..14.353 rows=3 loops=1)
Sort Key: (avg(((payload ->> 'total'::text))::numeric)) DESC
Sort Method: quicksort Memory: 25kB
Buffers: shared hit=1000
-> HashAggregate (cost=1800.53..1884.00 rows=5565 width=72) (actual time=14.314..14.342 rows=3 loops=1)
Group Key: (payload ->> 'payment_method'::text)
Batches: 1 Memory Usage: 217kB
Buffers: shared hit=1000


Step 2. Один запит двома способами.   
**Normalized-версія:**

In [23]:
%%sql
EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payment_method,
    AVG(total) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3;


 * postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw7_pg
16 rows affected.


QUERY PLAN
Limit (cost=269.06..269.07 rows=3 width=46) (actual time=8.985..8.987 rows=3 loops=1)
Buffers: shared hit=94
-> Sort (cost=269.06..269.07 rows=3 width=46) (actual time=8.983..8.984 rows=3 loops=1)
Sort Key: (avg(total)) DESC
Sort Method: quicksort Memory: 25kB
Buffers: shared hit=94
-> HashAggregate (cost=269.00..269.04 rows=3 width=46) (actual time=8.968..8.971 rows=3 loops=1)
Group Key: payment_method
Batches: 1 Memory Usage: 24kB
Buffers: shared hit=94


Порівняння підходів (JSONB vs Реляційний):   
\begin{array}{|c|c|c|c|}
\hline
\text
Метрика & JSONB-версія & Нормалізована\ версія & Ефект \\
& (t7\_hw\_events) & (t7\_hw\_checkouts\_norm) \\
\hline
Зчитано\ буферів\ (розмір) & 1000\ блоков\ (~8 МБ) & 94\ блока\ (~0.75 МБ) & У\ 10.6\ разів\ менше\ даних\ памяті \\
Час\ сканування\ таблиці & 8.858\ ms & 1.395\ ms & У\ 6.3\ разів\ быстріше\ читання\ строк  \\
Загальний\ час\ (Execution\ Time) & 14.452\ ms & 29.028\ ms & Швидше\ на\ 38\% \ на\ малому\ обсязі\\
Розрахункова\ вартість\ (Cost) & 1955.93 & 	269.07  & у\ 7.2\ рази\ менше\ навантаження\ на\ систему \\
\hline
\end{array}

# Reflection   
**1.Які total cost, actual time або ключові оператори плану ви побачили до і після GIN-індексу? Чи змінився Seq Scan на Bitmap Index Scan / Bitmap Heap Scan?**

*  Оператор Seq Scan гарантовано змінився на комбінацію Bitmap Index Scan та Bitmap Heap Scan. Замість повного перебору таблиці СУБД спочатку звернулася до GIN-індексу (Bitmap Index Scan), сформувала в пам'яті бітову карту ідентифікаторів рядків (TID), а потім вилучила дані з таблиці (Bitmap Heap Scan)
*   Total Cost (Загальна вартість) знизилася майже в 1.7 раза. До створення індексу розрахункова вартість становила 1758.22, а після підключення GIN упала до 1063.66. Це пов'язано з тим, що планувальник закладає значно менше CPU-витрат на обробку рядків.
*   Actual Time (реальний час виконання) скоротився більш ніж у 6.5 разів, а саме: з 47.818 ms до 7.196 ms.
*   Нюанс із буферами пам'яті (Buffers): До створення індексу Seq Scan прочитав усю таблицю (shared hit=1000). Після створення GIN-індексу сумарна кількість буферів склала 1004 (4 блоки самого індексу + 1000 блоків таблиці). Через те, що циклічно згенеровані дані рівномірно «розмазані» по сторінках, СУБД все одно довелося зайти в кожен із 1000 блоків таблиці. Однак час упав, оскільки база даних повністю уникнула CPU-витрат на парсинг і фільтрацію зайвих 47 500 документів — індекс одразу вказав на потрібні рядки.

**2.Чому GIN допомагає для @> containment, але не є найкращим рішенням для payload ->> 'device' = 'mobile'?**   

Індекс GIN (особливо з класом операторів jsonb_path_ops) влаштований як інвертований індекс шляхів і значень. Він бере внутрішню пару «шлях + значення» (наприклад, хеш від path: "referrer" + value: "google"), зберігає цей хеш у дерево один раз і пов'язує його зі списком ID рядків, де ця пара зустрічається.   

*   Для оператора **@> (containment)** запит знаходить рядки, де всередині документа Є точна структура {"referrer": "google"}». GIN миттєво знаходить хеш цієї пари у своєму дереві й видає готовий масив ID рядків.
*   Для оператора **->> (path-equality)** запит спочатку викликає функцію вилучення тексту (->>), яка має зайти в JSONB, розпарсити його, дістати значення за ключем device, повернути назовні плаский тип text, і тільки потім цей текст порівнюється через оператор = зі стрічкою 'mobile'. GIN-индекс не вміє індексувати результат виконання функцій або операторів рівності пласких типів. Він «заточений» виключно під структурну перевірку входження (@>, ?, ?&, ?|). Тому для ->> планувальник повністю ігнорує GIN і повертається до Seq Scan. Для таких задач ідеальним є Expression B-tree індекс, який заздалегідь обчислює текстовий результат функції для кожного рядка.   

**3.Який read/write trade-off створює GIN-індекс: що пришвидшується, а що може стати дорожчим?**    

Впровадження GIN-індексу на JSONB-колонку — це класичний компроміс між швидкістю читання та ціною підтримки даних. Read Trade-off прискорюється
Запити з оператором @> за будь-якими полями всередині JSONB (будь то payment_method, referrer, tags чи experiment) відпрацьовуватимуть за мілісекунди. При цьому не потрібно створювати десятки окремих індексів під кожне нове поле в документі.
Write & Storage Trade-off стає дорожчим, відбувається суттєве уповільнення INSERT / UPDATE / DELETE. При вставці або оновленні рядка з JSONB-документом PostgreSQL має розкласти цей документ на складові, розрахувати їхні хеші й оновити безліч внутрішніх вузлів GIN-індексу. Запис у таблиці з GIN-індексами може бути в кілька разів повільнішим, ніж у таблиці зі звичайними B-tree. Модифікація GIN-індексу генерує величезний обсяг WAL-логів (через спліти сторінок індексу). Процеси VACUUM також потребують значно більше ресурсів CPU та пам'яті для очищення й дефрагментації сторінок інвертованого індексу.
В цьому тесті індекс зайняв 1144 kB (~1.1 МБ) на всього 50 000 компактних рядків (близько 14% від обсягу таблиці). При зростанні обсягу даних до сотень мільйонів рядків і ускладненні структури JSONB, GIN-індекс почне потребувати гігабайти дискової та оперативної пам'яті для утримання робочих сторінок у кеші.   

**4.Коли у ML data infrastructure ви обрали б JSONB, а коли — нормалізовану таблицю?**

В ML-системах JSONB є незамінним на етапах дослідження (R&D), логування та збору сирих даних.
*   Логування передбачень моделей (Model Inference Logs), де передбачається логування кожного запиту до моделі та її відповідь. Різні версії моделей можуть повертати різну структуру даних, різну кількість скорів або нові метадані експериментів. Змінювати схему таблиці під кожен комміт ML-інженера — це архітектурна пастка. JSONB дозволяє записувати будь-які payload без зупинки продакшену.
*   Експерименти та Feature Engineering (A/B Тестування) на етапі гіпотез, коли дата-сайентисти постійно додають нові сирі ознаки (raw features). Швидше записати їх в один JSON-об'єкт, прогнати ноутбук, а вже після успішного тесту фіксувати схему.
*   Дані з зовнішніх API та вебхуків, де збирається контекст користувача, дані про трекінг кліків (click) або сторонні збагачувальні метрики, структура яких контролюється іншими розробниками.  

Нормалізовані пласкі таблиці ідеально підходять для Production Feature Stores, довгострокового зберігання ознак (Offline Features) та аналітики.
*   Feature Store низького часу затримки (Low-Latency Online Inference), коли модель у реальному часі запитує ознаки користувача (наприклад, total покупок за останні 30 днів для антифроду). Вилучення даних із пласкої таблиці не потребує парсингу JSON у CPU, дані лежать щільно (як продемонстровано у запитах: 94 блоки замість 1000), а стандартні B-tree індекси віддають рядок за субмілісекунди.
*   Масштабне навчання моделей (Offline Training Pipelines), де пайплайн викачує терабайти даних для навчання нової моделі. Читання пласкої таблиці заощадить тижні процесорного часу та терабайти мережевого трафіку, оскільки СУБД передає строго упаковані бінарні масиви без дублювання ключів текстового формату на кшталт "payment_method".
*   Строгі констреїнти та бізнес-метрики, фінансові або транзакційні ознаки (checkout, суми, кількості) тощо. ML-модель критична до якості даних: якщо замість INTEGER у полі кількості товарів запишеться стрічка, пайплайн навчання впаде. Пласкі таблиці з CHECK-констреїнтами гарантують валідність ознак на рівні бази даних.   

**5.Як би ви додали новий event-type refund у JSONB-підході та в normalized-підході? Які міграційні дії потрібні?**   

Якщо ми працюємо в JSONB-підході, додавання refund — це завдання на 1 хвилину для розробника ПЗ.
Якщо ми працюємо в нормалізованому підході, це повноцінна задача на проєктування схеми даних її звязків (кардинальностей), яка потребує створення нової таблиці, налаштування індексів та констреїнтів. Але в подальшому нормалізований підхід заощадить терабайти пам'яті та захистить модель від невалідних ознак.



